# YOLO26s — 지정 도면 맞춤 학습과 크기 대응

대상은 제공한 도면 한 장입니다. 같은 부품의 중복 캡처를 학습에서 제외하지 않습니다.

- 기존 50 epochs 가중치에서 추가 20 epochs 학습
- 1981장, 38클래스, 모두 640×640과 변환된 박스 좌표
- 확대 개별 심볼 + 기존 증강본 + 실제 주변 도면 캡처를 함께 학습
- 새로운 온라인 증강은 끔
- 잘림 권고 원본 2개와 연결 증강본 34장은 제외

검사 자료도 학습에 포함됩니다. 결과는 지정 도면 재검출 확인이며 새 도면 일반화 성능이 아닙니다.


## 실행 완료 결과 (2026-10-08)

YOLO26s를 기존 정상 50회 학습 모델에서 1,981장으로 추가 20회 학습했습니다.
새 온라인 증강 없이 기존 증강본과 도면 속 심볼 캡처를 함께 사용했습니다.

| 재검출 대상 | 이미지 | TP | FP | FN | mAP50 |
|---|---:|---:|---:|---:|---:|
| all38 | 73 | 73 | 2 | 0 | 99.500% |
| region4 | 21 | 36 | 0 | 0 | 99.500% |
| all_scene | 179 | 297 | 3 | 0 | 99.500% |

TP/FP/FN은 confidence 0.25, 같은 클래스끼리 정답박스 IoU 0.5 이상에서 집계합니다.
평가 입력과 실제 검출 입력은 모두 640×640, rect=False입니다.
이 평가 이미지들은 학습에 포함되었습니다. 해당 도면 재검출 결과이며 독립 검증이 아닙니다.

전체 PDF에서는 1,870개 창을 처리해 60개의 검출 후보를 얻었습니다.
전체 PDF 정답이 모두 라벨링된 것은 아니므로 후보 수가 정확도를 뜻하지 않습니다.
결과: training_yolo26_noaug_20261008/known_drawing_fit20_durable/RESULT.md
검수 화면: training_yolo26_noaug_20261008/known_drawing_fit20_durable/pdf_tiled_detection/review.html

원본 PDF는 로컬에서만 렌더링/검출했습니다. OpenAI API를 호출하지 않았습니다.
이번 단계에서는 CAD BLOCK 생성까지 실행하지 않았습니다.


## 640×640인데 왜 심볼 크기가 다른가?

640은 입력 이미지의 가로·세로입니다. 박스는 그 이미지 안에서 심볼이 차지하는 영역입니다.
같은 640 캔버스에서도 100픽셀 심볼과 541픽셀 심볼이 들어갈 수 있습니다.
박스 좌표 변환은 두 크기를 같게 만드는 것이 아니라 각각의 실제 위치/크기를 보존합니다.

이번 학습은 같은 클래스의 작은 도면 심볼과 큰 확대 캡처를 모두 포함합니다.
추론은 큰 PDF를 통째로 640으로 축소하지 않고 겹치는 타일로 검출합니다.
320/640/1280/2560픽셀 창을 각각 모델 입력 640으로 맞춰 작은 심볼과 큰 참조 형상을 함께 처리합니다.
원본 PDF의 선/곡선 및 CAD 좌표는 변경하지 않습니다.


In [ ]:
from pathlib import Path
import json,sys,subprocess
PROJECT=Path(r"C:/Users/Win11Pro/Documents/ChatGPT/기업프로젝트 cad")
WORK=PROJECT/"symbol_roi640"
PYTHON=PROJECT/".venv-ml/Scripts/python.exe"
RUN=WORK/"training_yolo26_noaug_20261008/known_drawing_fit20_durable"
print(json.loads((WORK/"known_drawing_training_20261008/summary.json").read_text(encoding="utf-8")))

## 학습 실행

완료된 실행이 있으면 재학습하지 않습니다. Windows 데이터 로더 workers=0, batch=16, BF16입니다. 실제 변환이 LetterBox와 Format만 있는지는 별도 검사 파일로 확인합니다.

In [ ]:
if (RUN/"result_summary.json").exists():
 print("완료된 실행 결과를 사용합니다.")
else:
 subprocess.run([str(PYTHON),"-u",str(WORK/"train_yolo26_noaug.py"),
   "--known-drawing","--epochs","20","--batch","16","--name","known_drawing_fit20_durable"],
   check=True,cwd=PROJECT)

## 클래스별 재검출 결과

all38/region4는 이전 시험과 같은 이미지로 비교합니다. all_scene은 도면 맥락 타일 179장 전체입니다. 이번에는 모두 학습에 포함된 자료입니다.

In [ ]:
import pandas as pd
from IPython.display import display,Markdown,Image as DisplayImage
result=json.loads((RUN/"result_summary.json").read_text(encoding="utf-8"))
display(pd.DataFrame(result["evaluations"]).T)
display(pd.read_csv(RUN/"reports/all_scene/per_class.csv").sort_values("mAP50"))
display(Markdown((RUN/"RESULT.md").read_text(encoding="utf-8")))
print(json.loads((RUN/"no_augmentation_audit.json").read_text(encoding="utf-8")))

## 정답박스와 예측박스 비교

왼쪽 정답 / 오른쪽 예측. 고정 confidence 0.25, 같은 클래스와 IoU 0.5를 기준으로 TP/FP/FN을 기록했습니다.

In [ ]:
rows=json.loads((RUN/"reports/all_scene/predictions_conf025.json").read_text(encoding="utf-8"))
for row in sorted(rows,key=lambda r:r["fn"]+r["fp"],reverse=True)[:6]:
 print(row["image"],"TP",row["tp"],"FP",row["fp"],"FN",row["fn"])
 display(DisplayImage(filename=str(RUN/"reports/all_scene"/row["preview"]),width=1100))

## 실제 PDF 타일 검출

로컬 Poppler로 렌더링한 전체 PDF PNG를 사용합니다. 작은/중간/큰 창에서 검출하고 박스를 원래 페이지 픽셀/PDF 포인트로 복원합니다. 타일 겹침의 중복 박스는 클래스별 NMS로 병합합니다. 출력 후보의 정확성은 review.html에서 도면과 함께 확인하세요.

In [ ]:
RASTER=WORK/"known_drawing_training_20261008/render/pdf_full_144dpi.png"
DETECTIONS=RUN/"pdf_tiled_detection"
if (DETECTIONS/"summary.json").exists():
 print(json.loads((DETECTIONS/"summary.json").read_text(encoding="utf-8")))
else:
 subprocess.run([str(PYTHON),"-u",str(WORK/"predict_drawing_tiles.py"),
   "--image",str(RASTER),"--weights",str(RUN/"weights/last.pt"),
   "--out",str(DETECTIONS),"--dpi","144","--conf","0.25"],check=True,cwd=PROJECT)
print("확대해서 볼 검출 후보:",DETECTIONS/"review.html")